# 09 · Reading/Writing Files and Pivot Tables

**Goal:** learn to load and save data in common formats (CSV, Excel, JSON), and reshape data
between "long" and "wide" formats using `pivot_table`, `pivot`, and `melt`.

### Writing and reading CSV files

CSV (Comma-Separated Values) is the most common data interchange format you'll encounter.

In [1]:
import pandas as pd

df = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "age": [25, 32, 18],
    "city": ["NYC", "LA", "Chicago"]
})

df.to_csv("people.csv", index=False)   # index=False avoids writing the row numbers as a column
print("Saved!")

loaded = pd.read_csv("people.csv")
print(loaded)

Saved!
      name  age     city
0    Alice   25      NYC
1      Bob   32       LA
2  Charlie   18  Chicago


### ⚠️ Always check `index=False` when writing, and inspect after reading

Forgetting `index=False` adds an extra unnamed column full of row numbers to your file.
Always sanity-check a freshly loaded file with `.head()` and `.dtypes`.

In [2]:
df.to_csv("people_with_index.csv")            # forgot index=False
print(pd.read_csv("people_with_index.csv"))     # notice the extra "Unnamed: 0" column!

print()
df.to_csv("people_no_index.csv", index=False)
print(pd.read_csv("people_no_index.csv"))         # clean -- no extra column

   Unnamed: 0     name  age     city
0           0    Alice   25      NYC
1           1      Bob   32       LA
2           2  Charlie   18  Chicago

      name  age     city
0    Alice   25      NYC
1      Bob   32       LA
2  Charlie   18  Chicago


### Useful `read_csv` options

Real-world CSVs are messy — different delimiters, extra header rows, specific columns needed,
particular dtypes. `read_csv` has parameters for all of this.

In [3]:
# Only load specific columns
print(pd.read_csv("people_no_index.csv", usecols=["name", "age"]))

# Force specific dtypes at load time
print(pd.read_csv("people_no_index.csv", dtype={"age": float}).dtypes)

# Parse a column as dates while reading
dates_df = pd.DataFrame({"event_date": ["2024-01-01", "2024-02-01"], "value": [10, 20]})
dates_df.to_csv("events.csv", index=False)

loaded_dates = pd.read_csv("events.csv", parse_dates=["event_date"])
print(loaded_dates.dtypes)

      name  age
0    Alice   25
1      Bob   32
2  Charlie   18
name        str
age     float64
city        str
dtype: object
event_date    datetime64[us]
value                  int64
dtype: object


### Excel files

Requires the `openpyxl` package to be installed (`pip install openpyxl`), but the pandas API
is identical in shape to the CSV functions.

In [4]:
try:
    df.to_excel("people.xlsx", index=False, sheet_name="People")
    loaded_excel = pd.read_excel("people.xlsx", sheet_name="People")
    print(loaded_excel)
except ImportError as e:
    print("openpyxl not installed -- run `pip install openpyxl` to enable Excel support.")
    print("Error detail:", e)

      name  age     city
0    Alice   25      NYC
1      Bob   32       LA
2  Charlie   18  Chicago


### JSON files

In [5]:
df.to_json("people.json", orient="records")
loaded_json = pd.read_json("people.json")
print(loaded_json)

      name  age     city
0    Alice   25      NYC
1      Bob   32       LA
2  Charlie   18  Chicago


### Quick reference: reading/writing functions

| Format | Write | Read |
|---|---|---|
| CSV | `df.to_csv(path)` | `pd.read_csv(path)` |
| Excel | `df.to_excel(path)` | `pd.read_excel(path)` |
| JSON | `df.to_json(path)` | `pd.read_json(path)` |
| SQL | `df.to_sql(...)` | `pd.read_sql(...)` |

## Reshaping data: long vs. wide format

- **Wide format**: one row per subject, one column per variable/time period (good for reading).
- **Long format**: one row per (subject, variable) combination (good for analysis/plotting).

Understanding this distinction is essential for using `pivot`/`pivot_table` and `melt`
correctly.

In [6]:
long_df = pd.DataFrame({
    "student": ["Alice", "Alice", "Bob", "Bob"],
    "subject": ["Math", "Science", "Math", "Science"],
    "score": [88, 92, 79, 85]
})
print("LONG format:")
print(long_df)

LONG format:
  student  subject  score
0   Alice     Math     88
1   Alice  Science     92
2     Bob     Math     79
3     Bob  Science     85


### `.pivot()` — long to wide (no aggregation, requires unique combinations)

Use `.pivot()` when each (index, columns) combination appears exactly once — it just
reshapes, without needing to aggregate anything.

In [7]:
wide_df = long_df.pivot(index="student", columns="subject", values="score")
print("WIDE format:")
print(wide_df)

WIDE format:
subject  Math  Science
student               
Alice      88       92
Bob        79       85


### `.pivot_table()` — long to wide, WITH aggregation (handles duplicates)

Use `pivot_table` (seen briefly in notebook 06) when there might be multiple rows for the same
(index, columns) combination — it aggregates them (default: mean).

In [8]:
long_with_dupes = pd.DataFrame({
    "student": ["Alice", "Alice", "Alice", "Bob", "Bob"],
    "subject": ["Math", "Math", "Science", "Math", "Science"],
    "score": [88, 90, 92, 79, 85]     # Alice has TWO Math scores
})

# .pivot() would fail here (duplicate index/column combination) -- pivot_table aggregates instead
pivoted = long_with_dupes.pivot_table(index="student", columns="subject", values="score", aggfunc="mean")
print(pivoted)

subject  Math  Science
student               
Alice    89.0     92.0
Bob      79.0     85.0


### `.melt()` — wide to long (the inverse of pivot)

Use `.melt()` when you have a wide table and need to "unpivot" it back into long format —
often required before plotting with libraries that expect long-format data.

In [9]:
wide = pd.DataFrame({
    "student": ["Alice", "Bob"],
    "Math": [88, 79],
    "Science": [92, 85]
})
print("WIDE:")
print(wide)

melted = wide.melt(id_vars="student", var_name="subject", value_name="score")
print()
print("LONG (melted):")
print(melted)

WIDE:
  student  Math  Science
0   Alice    88       92
1     Bob    79       85

LONG (melted):
  student  subject  score
0   Alice     Math     88
1     Bob     Math     79
2   Alice  Science     92
3     Bob  Science     85


### `pivot_table` with multiple aggregations and margins

In [10]:
sales = pd.DataFrame({
    "region": ["East", "East", "West", "West", "East", "West"],
    "product": ["A", "B", "A", "B", "A", "B"],
    "revenue": [100, 150, 200, 130, 120, 170]
})

pivot = sales.pivot_table(
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    margins=True,          # adds row/column totals, labeled "All"
    margins_name="Total"
)
print(pivot)

product    A    B  Total
region                  
East     220  150    370
West     200  300    500
Total    420  450    870


### 🧠 Quick check

1. Why should you usually pass `index=False` when writing a DataFrame to CSV?
2. What's the difference between `.pivot()` and `.pivot_table()`?
3. What does `.melt()` do, conceptually?

<details>
<summary>Answers</summary>

1. Otherwise pandas writes the row index as an extra unnamed column, which is rarely wanted
   when the DataFrame already has a plain sequential index.
2. `.pivot()` requires every (index, column) combination to be unique — it purely reshapes.
   `.pivot_table()` handles duplicate combinations by aggregating them (e.g. taking the mean).
3. It converts a "wide" table (one column per variable) into "long" format (one row per
   observation) — the inverse operation of pivoting.
</details>

### ✍️ Practice

1. Save a small DataFrame to CSV, then reload it and confirm the dtypes match what you'd
   expect (watch out for dates loading as plain strings unless you use `parse_dates`).
2. Create a long-format DataFrame of monthly sales per region, and pivot it to wide format
   (regions as rows, months as columns).
3. Melt a wide-format DataFrame of quarterly revenue per product back into long format.
4. Build a `pivot_table` with `margins=True` to get row and column totals for a sales dataset.

Continue to **`10_practice_project.ipynb`** to combine everything you've learned.